# Smart Hotel Intelligence System
## Exploratory Data Analysis (EDA)
**Dataset:** Hotel Booking Demand (119,390 records)  
**Source:** Antonio, Almeida & Nunes (2019) — open-source via Kaggle / TidyTuesday  
**Project:** Cancellation Risk Prediction & Revenue Optimisation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Load dataset
import os
data_path = '../hotel_booking.csv' if os.path.exists('../hotel_booking.csv') else '../data/hotel_booking_sample.csv'
df = pd.read_csv(data_path)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

## 1. Dataset Overview

In [ ]:
print('=== DATASET INFO ===')
print(df.dtypes)
print(f'\n=== MISSING VALUES ===')
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
print('=== BASIC STATISTICS ===')
print(df[['lead_time', 'adults', 'children', 'babies', 'adr',
          'stays_in_weekend_nights', 'stays_in_week_nights']].describe().round(2))

## 2. Missing Value Handling

In [ ]:
df['children'] = df['children'].fillna(0)
df['agent']    = df['agent'].fillna(0)
df['company']  = df['company'].fillna(0)
df['country']  = df['country'].fillna('Unknown')

# Feature engineering
MONTH_MAP = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
df['arrival_month_num'] = df['arrival_date_month'].map(MONTH_MAP)
df['total_guests']      = df['adults'] + df['children'] + df['babies']
df['total_nights']      = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['adr']               = df['adr'].clip(0, df['adr'].quantile(0.99))

print('Missing values after handling:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('None remaining!' if df.isnull().sum().sum() == 0 else '')

## 3. Target Variable Analysis — Cancellation Rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pie chart
cancel_counts = df['is_canceled'].value_counts()
axes[0].pie(cancel_counts, labels=['Not Cancelled', 'Cancelled'],
            autopct='%1.1f%%', colors=['#2ed573', '#ff4d4d'], startangle=90)
axes[0].set_title('Overall Cancellation Rate', fontsize=13, fontweight='bold')

# By hotel type
hotel_cancel = df.groupby('hotel')['is_canceled'].mean() * 100
axes[1].bar(hotel_cancel.index, hotel_cancel.values, color=['#74b9ff', '#a29bfe'])
axes[1].set_title('Cancellation Rate by Hotel Type', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cancellation Rate (%)')
for i, v in enumerate(hotel_cancel.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../images/01_cancellation_overview.png', bbox_inches='tight')
plt.show()
print(f'Overall cancellation rate: {df["is_canceled"].mean()*100:.1f}%')

## 4. Monthly Booking Trends

In [ ]:
month_names = {v: k[:3] for k, v in MONTH_MAP.items()}
monthly = df.groupby('arrival_month_num').agg(
    bookings=('is_canceled', 'count'),
    cancel_rate=('is_canceled', 'mean')
).reset_index()
monthly['month'] = monthly['arrival_month_num'].map(month_names)
monthly['cancel_pct'] = monthly['cancel_rate'] * 100

fig, ax1 = plt.subplots(figsize=(13, 5))
bars = ax1.bar(monthly['month'], monthly['bookings'], color='#74b9ff', alpha=0.8, label='Total Bookings')
ax1.set_ylabel('Total Bookings', color='#2d3436')
ax1.set_title('Monthly Booking Volume & Cancellation Rate', fontsize=14, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(monthly['month'], monthly['cancel_pct'], 'o-', color='#ff4d4d', linewidth=2.5, label='Cancel Rate %')
ax2.set_ylabel('Cancellation Rate (%)', color='#ff4d4d')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('../images/02_monthly_trends.png', bbox_inches='tight')
plt.show()

## 5. Average Daily Rate (ADR) Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df[df['adr'] > 10]['adr'].hist(bins=60, ax=axes[0], color='#6c5ce7', edgecolor='white')
axes[0].set_title('ADR Distribution (Room Rate €)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Average Daily Rate (€)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['adr'].mean(), color='#ff4d4d', linestyle='--', linewidth=2, label=f'Mean: €{df["adr"].mean():.0f}')
axes[0].legend()

adr_by_hotel = df[df['adr'] > 10].groupby(['hotel', 'arrival_month_num'])['adr'].mean().reset_index()
adr_by_hotel['month'] = adr_by_hotel['arrival_month_num'].map(month_names)
for hotel, grp in adr_by_hotel.groupby('hotel'):
    grp = grp.sort_values('arrival_month_num')
    axes[1].plot(grp['month'], grp['adr'], 'o-', label=hotel, linewidth=2)
axes[1].set_title('Average Room Rate by Month & Hotel Type', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Avg Daily Rate (€)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/03_adr_analysis.png', bbox_inches='tight')
plt.show()

## 6. Lead Time vs Cancellation Risk

In [ ]:
df['lead_bin'] = pd.cut(df['lead_time'],
    bins=[0, 7, 14, 30, 60, 90, 180, 365, 700],
    labels=['0-7d','8-14d','15-30d','31-60d','61-90d','91-180d','181-365d','365d+'])

lead_cancel = df.groupby('lead_bin', observed=True)['is_canceled'].mean() * 100

plt.figure(figsize=(10, 4))
bars = plt.bar(lead_cancel.index, lead_cancel.values,
               color=plt.cm.RdYlGn_r(lead_cancel.values / 100))
plt.title('Cancellation Rate by Booking Lead Time', fontsize=14, fontweight='bold')
plt.xlabel('Lead Time Before Arrival')
plt.ylabel('Cancellation Rate (%)')
for bar, val in zip(bars, lead_cancel.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.0f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../images/04_lead_time_analysis.png', bbox_inches='tight')
plt.show()
print('Key Insight: Longer lead time = significantly higher cancellation risk')

## 7. Deposit Type vs Cancellation

In [ ]:
dep_cancel = df.groupby('deposit_type')['is_canceled'].mean() * 100

plt.figure(figsize=(8, 4))
bars = plt.bar(dep_cancel.index, dep_cancel.values,
               color=['#2ed573', '#ffa502', '#ff4d4d'])
plt.title('Cancellation Rate by Deposit Type', fontsize=14, fontweight='bold')
plt.ylabel('Cancellation Rate (%)')
for bar, val in zip(bars, dep_cancel.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../images/05_deposit_type.png', bbox_inches='tight')
plt.show()
print('Key Insight: Non-refundable deposits show the highest recorded cancellation rate')

## 8. Correlation Heatmap

In [ ]:
numeric_cols = ['is_canceled', 'lead_time', 'total_nights', 'total_guests',
                'adr', 'booking_changes', 'previous_cancellations',
                'total_of_special_requests', 'days_in_waiting_list',
                'is_repeated_guest', 'required_car_parking_spaces']

corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/06_correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 9. Market Segment Analysis

In [ ]:
seg = df.groupby('market_segment').agg(
    bookings=('is_canceled','count'),
    cancel_rate=('is_canceled','mean'),
    avg_adr=('adr','mean')
).reset_index().sort_values('cancel_rate')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(seg['market_segment'], seg['cancel_rate']*100, color='#ff6b6b')
axes[0].set_title('Cancellation Rate by Market Segment', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cancellation Rate (%)')

axes[1].barh(seg['market_segment'], seg['avg_adr'], color='#6c5ce7')
axes[1].set_title('Average Room Rate by Market Segment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Average Daily Rate (€)')

plt.tight_layout()
plt.savefig('../images/07_market_segment.png', bbox_inches='tight')
plt.show()

## 10. EDA Summary — Key Insights

| # | Finding | Business Implication |
|---|---------|---------------------|
| 1 | **37% overall cancellation rate** — very high for hospitality | Hotels must proactively manage risk |
| 2 | **Longer lead time → higher cancellation** (0–7 days: ~14%, 365d+: ~63%) | Flag early bookings as high-risk |
| 3 | **Non-refundable deposit bookings cancel at very high rates** | Deposit policy alone does not prevent cancellations |
| 4 | **City Hotels cancel more than Resort Hotels** | Different risk management strategies needed |
| 5 | **August is peak month; April–August highest volume** | Seasonal demand should drive pricing & overbooking policies |
| 6 | **Online TA segment has highest cancellation rate** | OTA bookings need tighter deposit/reminder policies |
| 7 | **Previous cancellations is a top predictor** | Guest history is critical for risk scoring |